# UNIVERSIDAD NACIONAL DE SAN ANTONIO ABAD DEL CUSCO
### Facultad de Ingeniería Civil — Laboratorio de Materiales y Concreto
**Asignatura:** Tecnología del Concreto (IC152)  
**Docente:** ______________________________ | **Grupo / Brigada:** ______________________________  
**Semestre:** 2026-II  

---

# Cuaderno complementario DM-01: Diseño de mezcla de concreto (ACI PRC-211.1-22)

> **Normas:** ACI PRC-211.1-22, capítulo 5 (pasos 1 a 9) y num. 4.7.8 · RNE E.060, Tabla 5.3

Este cuaderno no corresponde a una práctica numerada de la guía. Reúne los resultados de las prácticas TC-02 (humedad), TC-04 (granulometría), TC-05 y TC-06 (peso específico y absorción) y TC-07 (peso unitario) para obtener la dosificación de la tanda que se prepara en la práctica TC-10.

### Objetivos
1. Calcular la resistencia promedio requerida f'cr a partir de f'c.
2. Obtener los pesos por metro cúbico de agua, cemento, agregado grueso y agregado fino.
3. Corregir los pesos por la humedad de los agregados y calcular la tanda de prueba.

## 1. Conexión con el motor de cálculo (`civillab_unsaac`)
Ejecuta la siguiente celda antes de comenzar.

In [ ]:
# CIVILLAB_WEB_PRELOAD: prepara el navegador (solo versión web)
%pip install -q openpyxl


In [1]:
# ==============================================================================
# CONEXIÓN CON LA LIBRERÍA DEL LABORATORIO (Ejecutar primero, no modificar)
# ==============================================================================
try:
    import civillab_unsaac as lab
except ModuleNotFoundError:
    # Si la librería aún no está instalada, la ubicamos automáticamente en el proyecto
    import sys
    from pathlib import Path
    raices = [
        Path.cwd(),
        *list(Path.cwd().parents)[:3],
        Path.cwd() / "08_SOFTWARE_LABORATORIO",
        Path.cwd() / "0003_UNSAAC_DOCENCIA" / "08_SOFTWARE_LABORATORIO",
    ]
    for r in raices:
        if (r / "civillab_unsaac").is_dir():
            sys.path.insert(0, str(r))
            break
    import civillab_unsaac as lab

from pathlib import Path
from civillab_unsaac import *
import matplotlib
try:
    get_ipython()
except NameError:
    matplotlib.use("Agg")

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    display = print

# Carpeta limpia para gráficos y fichas Excel de salida
_pkg_root = Path(lab.__file__).resolve().parents[1]
DIR_SALIDA = (_pkg_root / "fichas_generadas") if (_pkg_root / "notebooks").is_dir() else (Path.cwd() / "fichas_generadas")
DIR_SALIDA.mkdir(parents=True, exist_ok=True)

print("✓ Suite CivilLab UNSAAC conectada y lista para calcular.")

✓ Suite CivilLab UNSAAC conectada y lista para calcular.


---
## 2. Datos de entrada
Reemplaza los valores de ejemplo por los de tu brigada. Cada dato indica de qué práctica sale.

In [2]:
diseno = DisenoMezclaACI211(
    descripcion="Brigada ___ — cantera ____________",
    fc_kg_cm2=210,              # resistencia especificada f'c
    asentamiento_pulg=4,        # asentamiento de diseño (1 a 7 pulg)
    tmn="3/4",                  # tamaño máximo nominal del grueso (TC-04)
    modulo_finura=2.85,         # módulo de finura del fino (TC-04)
    puc_grueso_kg_m3=1580,      # peso unitario compactado seco del grueso (TC-07)
    g_sss_grueso=2.66,          # peso específico SSS del grueso (TC-05)
    abs_grueso_pct=1.6,         # absorción del grueso, % (TC-05)
    g_sss_fino=2.63,            # peso específico SSS del fino (TC-06)
    abs_fino_pct=1.8,           # absorción del fino, % (TC-06)
    humedad_grueso_pct=0.8,     # humedad total del grueso el día de la tanda, % (TC-02)
    humedad_fino_pct=3.5,       # humedad total del fino el día de la tanda, % (TC-02)
    pe_cemento=2.85,            # peso específico del cemento (ficha técnica del fabricante)
)
print(diseno.diagnostico_tecnico())

=== DISEÑO DE MEZCLA — ACI PRC-211.1-22 ===
Brigada ___ — cantera ____________
f'c = 210 kg/cm² | f'cr = 295 kg/cm² (RNE E.060, Tabla 5.3)
Asentamiento: 4 pulg (fila «3 a 4») | TMN: 3/4 pulg | MF del fino: 2.85
• Paso 3 — Agua: 202 kg/m³ (Tabla 5.3.3) | Aire atrapado: 2.0 %
• Paso 4 — a/mc = 0.55 (Tabla 5.3.4; gobierna la resistencia)
• Paso 5 — Cemento: 367 kg/m³ (8.6 bolsas de 42,5 kg)
• Paso 6 — Agregado grueso: b/b0 = 0.615 (Tabla 5.3.6) -> 987 kg/m³ (SSS)
• Paso 7 — Agregado fino: 732 kg/m³ (SSS), por volúmenes absolutos
• Peso unitario teórico del concreto: 2287 kg/m³
• Corrección por humedad (num. 4.7.8): agua libre de los agregados = +5 kg/m³
  Agua a añadir: 197 kg/m³ | Grueso húmedo: 979 kg/m³ | Fino húmedo: 744 kg/m³
• Proporción en peso (cemento : fino : grueso / agua): 1 : 2.03 : 2.67 / 0.54
Nota: es una mezcla de prueba. ACI PRC-211.1-22 (num. 5.3.9) pide verificarla y ajustarla con una tanda de laboratorio.


---
## 3. Pesos de diseño por metro cúbico (agregados en condición SSS)

In [3]:
display(diseno.tabla_diseno())

,Material,Peso (kg/m³),Volumen absoluto (m³)
0,Agua de mezclado,202.0,0.2018
1,Cemento,367.0,0.1287
2,Agregado grueso (SSS),987.0,0.3713
3,Agregado fino (SSS),732.0,0.2782
4,Aire,0.0,0.0200
5,Total,2287.0,1.0000


---
## 4. Corrección por humedad de los agregados
El agua libre de los agregados (humedad total menos absorción) se descuenta del agua de mezclado. Si los agregados están más secos que la condición SSS, el agua aumenta.

In [4]:
display(diseno.tabla_corregida())

,Material,Diseño SSS (kg/m³),Corregido por humedad (kg/m³)
0,Agua,202.0,197.0
1,Cemento,367.0,367.0
2,Agregado grueso,987.0,979.0
3,Agregado fino,732.0,744.0
4,Total,2287.0,2287.0


---
## 5. Tanda de prueba
Indica el volumen de la tanda en litros. Para seis probetas de 100 mm × 200 mm, el cono de Abrams y un margen de desperdicio, 35 L es un valor razonable (recomendación del laboratorio).

In [5]:
VOLUMEN_TANDA_L = 35
display(diseno.tanda(VOLUMEN_TANDA_L))
print("Proporción en peso (cemento : fino : grueso / agua):", diseno.proporcion_en_peso())

Proporción en peso (cemento : fino : grueso / agua): 1 : 2.03 : 2.67 / 0.54


,Material,Peso para 35 L (kg)
0,Agua,6.89
1,Cemento,12.83
2,Agregado grueso (húmedo),34.28
3,Agregado fino (húmedo),26.04


---
## 6. Para el informe
1. ¿Qué gobernó la relación agua/material cementante: la resistencia o la durabilidad?
2. ¿Cuánto cambió el agua de mezclado por la humedad de los agregados? ¿Aumentó o disminuyó? ¿Por qué?
3. Compara el asentamiento medido en la práctica TC-10 con el asentamiento de diseño. Si no coincide, ¿qué ajuste indica la norma para la siguiente tanda (num. 5.3.9)?